In [5]:
import re
from os import listdir
from os.path import isfile, join, exists
import tempfile
import csv
import shutil
import pandas as pd  

def fix_row(row):
    return [f'"{field}"' if ',' in field and not (field.startswith('"') and field.endswith('"')) else field for field in row]

path = "../data"
path_clean = "../data/clean"

files = [re.sub(r".csv", "", f) for f in listdir(path) if isfile(join(path, f))]

for file in files:
    full_path = f"{path}/{file}.csv"
    if not exists(full_path):
        print(f"File not found: {full_path}")

    with open(full_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        lines = list(reader)

    header = lines[0]
    fixed_rows = [header]

    for i, row in enumerate(lines[1:], start=2):  # start=2 to match line numbers (1-based with header)
        if len(row) != 12:
            # Try to auto-fix by quoting comma-containing fields
            row = fix_row(row)
        if len(row) == 12:
            fixed_rows.append(row)
        else:
            print(f"Skipping malformed line {i} in {file}: {row}")

    # Save to a temporary cleaned file

    with tempfile.NamedTemporaryFile(mode="w+", newline='', delete=False, suffix=".csv") as tmp:
        writer = csv.writer(tmp)
        writer.writerows(fixed_rows)
        tmp_path = tmp.name

    # Save permanently with the same name as original
    cleaned_path = join(path_clean, f"{file}.csv")
    shutil.copy(tmp_path, cleaned_path)
    print(f"Saved cleaned file: {path_clean}")
        

Saved cleaned file: ../data/clean


In [ ]:
# Only needed one
'''
import requests
import time
import json

with open("occu.json", "r", encoding="utf-8") as f:
    occu_data = json.load(f)

occupations = list(occu_data["occu"].keys())
organizations = list(occu_data["orga"].keys())

occ_str = ",".join([f"wd:{qid}" for qid in occupations])
org_str = ",".join([f"wd:{qid}" for qid in organizations])
unmatched_entities = [] 

# Load the dataset
books = pd.read_csv(f"{path_clean}/books.csv")

# Normalize author names: collapse multiple spaces into one
def clean_author(author):
    return re.sub(r'\s+', ' ', author.strip())

books["authors"] = books["authors"].astype(str).apply(clean_author)

# Extract unique author names
unique_authors = set()
for author_list in books["authors"]:
    for author in author_list.split("/"):
        unique_authors.add(clean_author(author))

# Extract Wikipedia URL from Wikidata entity
def get_wikipedia_from_entity(entity_id):
    try:
        url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
        response = requests.get(url, timeout=20)
        entity_data = response.json()
        sitelinks = entity_data["entities"][entity_id]["sitelinks"]
        if "enwiki" in sitelinks:
            return sitelinks["enwiki"]["url"]
    except Exception as e:
        print(f"[Wiki URL fetch error] {entity_id}: {e}")
    return ""

def validate_occupation(entity_id):    
    query = f"""
    ASK {{
      {{
        wd:{entity_id} wdt:P106 ?occupation .
        FILTER(?occupation IN ({occ_str}))
      }} UNION {{
        wd:{entity_id} wdt:P31 ?type .
        FILTER(?type IN ({org_str}))
      }}
    }}
    """
    time.sleep(2)
    try:
        response = requests.get(
            "https://query.wikidata.org/sparql",
            headers={"Accept": "application/sparql-results+json"},
            params={"query": query},
            timeout=20
        )
        result = response.json().get("boolean", False)
        print(f"{entity_id} {result}")
        return result
    except Exception as e:
        print(f"[Occupation check error] {entity_id}: {e}")
    return False

# Get Wikidata + Wikipedia URL for author
def get_wikidata_url(author_name):
    author_name = clean_author(author_name)

    # --- Primary SPARQL method ---
    query = f"""    
    SELECT ?entity ?article WHERE {{
      {{
        ?entity rdfs:label "{author_name}"@en .
        ?entity wdt:P106 ?occupation .
        FILTER(?occupation IN ({occ_str}))
      }} UNION {{
        ?entity rdfs:label "{author_name}"@en .
        ?entity wdt:P31 ?type .
        FILTER(?type IN ({org_str}))
      }}
      OPTIONAL {{
        ?article schema:about ?entity ;
                 schema:isPartOf <https://en.wikipedia.org/> .
      }}
    }} LIMIT 1
    """
    try:
        response = requests.get(
            "https://query.wikidata.org/sparql",
            headers={"Accept": "application/sparql-results+json"},
            params={"query": query},
            timeout=20
        )
        data = response.json()
        bindings = data.get("results", {}).get("bindings", [])
        if bindings:
            entity_url = bindings[0]["entity"]["value"]
            wikipedia_url = bindings[0].get("article", {}).get("value", "")
            return entity_url, wikipedia_url
    except Exception as e:
        print(f"[SPARQL error] {author_name}: {e}")

    # --- Fallback: Wikidata API Search ---
    try:
        search_url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbsearchentities',
            'language': 'en',
            'format': 'json',
            'search': author_name
        }
        response = requests.get(search_url, params=params, timeout=20)
        data = response.json()
        for item in data.get("search", []):
            entity_id = item.get("id")
            if validate_occupation(entity_id):
                wikidata_url = f"https://www.wikidata.org/entity/{entity_id}"
                wikipedia_url = get_wikipedia_from_entity(entity_id)
                return wikidata_url, wikipedia_url
    except Exception as e:
        print(f"[Fallback error] {author_name}: {e}")

    return "", ""

# Query and store results
results = []
for author in sorted(unique_authors):
    print(f"Checking: {author}")
    wikidata_url, wikipedia_url = get_wikidata_url(author)
    results.append({
        "author": author,
        "wikidata_url": wikidata_url,
        "wikipedia_url": wikipedia_url
    })
    df = pd.DataFrame(results)
    df.to_csv(join(path_clean, "authors.csv"), index=False)
    time.sleep(3)

# Save to CSV

print("Saved authors_wikidata.csv.")
'''


<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_123680/1191128101.py:2: SyntaxWarning: invalid escape sequence '\s'
  '''


'\nimport requests\nimport time\nimport json\n\nwith open("occu.json", "r", encoding="utf-8") as f:\n    occu_data = json.load(f)\n\noccupations = list(occu_data["occu"].keys())\norganizations = list(occu_data["orga"].keys())\n\nocc_str = ",".join([f"wd:{qid}" for qid in occupations])\norg_str = ",".join([f"wd:{qid}" for qid in organizations])\nunmatched_entities = [] \n\n# Load the dataset\nbooks = pd.read_csv(f"{path_clean}/books.csv")\n\n# Normalize author names: collapse multiple spaces into one\ndef clean_author(author):\n    return re.sub(r\'\\s+\', \' \', author.strip())\n\nbooks["authors"] = books["authors"].astype(str).apply(clean_author)\n\n# Extract unique author names\nunique_authors = set()\nfor author_list in books["authors"]:\n    for author in author_list.split("/"):\n        unique_authors.add(clean_author(author))\n\n# Extract Wikipedia URL from Wikidata entity\ndef get_wikipedia_from_entity(entity_id):\n    try:\n        url = f"https://www.wikidata.org/wiki/Special:

In [7]:
files_clean = [re.sub(r".csv", "", f) for f in listdir(path_clean) if isfile(join(path_clean, f))]
data_dict = {}
print(files_clean)

for file in files_clean:
    try:
        data_dict[file] = pd.read_csv(join(path_clean,f"{file}.csv"))
    except Exception as e:
        print(f"Failed to read cleaned CSV for {file}: {e}")

for key, value in data_dict.items():
    print(key)
    print(value)
    print("---------")

['books', 'authors']
books
       bookID                                              title  \
0           1  Harry Potter and the Half-Blood Prince (Harry ...   
1           2  Harry Potter and the Order of the Phoenix (Har...   
2           4  Harry Potter and the Chamber of Secrets (Harry...   
3           5  Harry Potter and the Prisoner of Azkaban (Harr...   
4           8  Harry Potter Boxed Set Books 1-5 (Harry Potter...   
...       ...                                                ...   
11122   45631   Expelled from Eden: A William T. Vollmann Reader   
11123   45633                        You Bright and Risen Angels   
11124   45634                    The Ice-Shirt (Seven Dreams #1)   
11125   45639                                        Poor People   
11126   45641                        Las aventuras de Tom Sawyer   

                                                 authors  average_rating  \
0                             J.K. Rowling/Mary GrandPré            4.57   
1   